# 06. Dependency Scanning & CVE Detection

## Background

Modern applications pull in hundreds of open-source packages. Each dependency is a potential attack surface: a single vulnerable transitive dependency can expose your entire application. The 2021 Log4Shell vulnerability (CVE-2021-44228) showed this starkly — millions of applications were vulnerable because they transitively depended on log4j-core, often without knowing it.

Dependency scanning maintains a Software Composition Analysis (SCA) view of your supply chain: what packages are in use, what versions, and whether any known CVEs affect them. Tools like pip-audit, Safety, Snyk, and GitHub Dependabot automate this in CI pipelines.

## What You'll Learn

- How CVE databases (NVD, OSV.dev) are structured and queried
- Building a transitive dependency graph and finding vulnerable nodes
- CVSS scoring: base, temporal, and environmental metrics
- License compliance scanning alongside security scanning

## Why This Matters

The average Python ML project has 50+ direct dependencies and 200+ transitive ones. Attackers actively target popular ML libraries: malicious packages impersonating `torch`, `transformers`, and `numpy` have appeared on PyPI. Automated dependency scanning is the first line of defense against supply chain compromise in the AI/ML ecosystem.


## CVE Database & CVSS Scoring

In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Set
from enum import Enum
import json

class CVSSSeverity(Enum):
    CRITICAL = 'CRITICAL'  # 9.0-10.0
    HIGH = 'HIGH'          # 7.0-8.9
    MEDIUM = 'MEDIUM'      # 4.0-6.9
    LOW = 'LOW'            # 0.1-3.9
    NONE = 'NONE'          # 0.0

@dataclass
class CVE:
    cve_id: str
    description: str
    cvss_score: float
    affected_package: str
    affected_versions: str  # e.g. '<2.1.0' or '>=1.0,<1.5'
    fixed_version: Optional[str]
    published_date: str

    @property
    def severity(self) -> CVSSSeverity:
        if self.cvss_score >= 9.0: return CVSSSeverity.CRITICAL
        elif self.cvss_score >= 7.0: return CVSSSeverity.HIGH
        elif self.cvss_score >= 4.0: return CVSSSeverity.MEDIUM
        elif self.cvss_score > 0.0: return CVSSSeverity.LOW
        return CVSSSeverity.NONE

# Simulated CVE database (subset relevant to ML/LLM apps)
CVE_DATABASE: List[CVE] = [
    CVE('CVE-2023-36189', 'LangChain SQLDatabase arbitrary code execution via SQL injection',
        9.8, 'langchain', '<0.0.247', '0.0.247', '2023-06-23'),
    CVE('CVE-2023-29197', 'Gradio arbitrary file read via path traversal',
        7.5, 'gradio', '<3.32.0', '3.32.0', '2023-04-19'),
    CVE('CVE-2023-26141', 'Sidekick: Prototype pollution via merge',
        6.5, 'sidekick', '<0.3.2', '0.3.2', '2023-03-15'),
    CVE('CVE-2022-21699', 'IPython arbitrary code execution via untrusted notebooks',
        8.8, 'ipython', '<7.31.1', '7.31.1', '2022-01-19'),
    CVE('CVE-2023-46137', 'Twisted HTTP request smuggling',
        5.3, 'twisted', '<23.10.0', '23.10.0', '2023-10-25'),
    CVE('CVE-2024-0232', 'SQLite heap-buffer overflow in jsonb',
        5.9, 'sqlite', '<3.43.2', '3.43.2', '2024-01-16'),
    CVE('CVE-2023-37276', 'aiohttp HTTP request smuggling',
        5.9, 'aiohttp', '<3.8.5', '3.8.5', '2023-07-19'),
    CVE('CVE-2023-41419', 'Gevent monkey-patch allows SSRF',
        9.1, 'gevent', '<23.9.0', '23.9.0', '2023-09-19'),
]

print(f'CVE database loaded: {len(CVE_DATABASE)} entries')
print(f'Severity distribution:')
from collections import Counter
dist = Counter(c.severity.value for c in CVE_DATABASE)
for sev in ['CRITICAL','HIGH','MEDIUM','LOW']:
    print(f'  {sev}: {dist.get(sev, 0)}')


## Dependency Graph & Transitive Scanning

In [ ]:
# Simulated project dependencies (direct + transitive)
DEPENDENCY_GRAPH = {
    'myapp': ['langchain==0.0.200', 'gradio==3.20.0', 'fastapi==0.95.0', 'aiohttp==3.8.3'],
    'langchain==0.0.200': ['requests==2.28.0', 'pydantic==1.10.0', 'sqlalchemy==2.0.0'],
    'gradio==3.20.0': ['aiohttp==3.8.3', 'fastapi==0.95.0', 'websockets==11.0'],
    'fastapi==0.95.0': ['starlette==0.27.0', 'pydantic==1.10.0'],
    'aiohttp==3.8.3': ['multidict==6.0.0', 'yarl==1.9.0'],
    'requests==2.28.0': ['certifi==2023.5.7', 'urllib3==1.26.0'],
}

def parse_pkg_version(dep: str):
    if '==' in dep: return dep.split('==')[0], dep.split('==')[1]
    return dep, '0.0.0'

def get_all_dependencies(graph: dict, root: str = 'myapp') -> Set[str]:
    '''BFS to collect all transitive dependencies.'''
    visited = set()
    queue = [root]
    while queue:
        node = queue.pop(0)
        if node in visited: continue
        visited.add(node)
        for dep in graph.get(node, []):
            queue.append(dep)
    visited.discard(root)
    return visited

def is_version_affected(installed: str, affected_spec: str) -> bool:
    '''Simple version comparison — production would use packaging.version.'''
    if affected_spec.startswith('<'):
        threshold = affected_spec[1:]
        return tuple(int(x) for x in installed.split('.')) < tuple(int(x) for x in threshold.split('.'))
    return False

def scan_dependencies(graph: dict) -> List[Dict]:
    all_deps = get_all_dependencies(graph)
    all_deps.update(graph.get('myapp', []))
    findings = []

    for dep in all_deps:
        pkg, version = parse_pkg_version(dep)
        for cve in CVE_DATABASE:
            if cve.affected_package.lower() == pkg.lower():
                if is_version_affected(version, cve.affected_versions):
                    findings.append({
                        'package': pkg, 'version': version,
                        'cve': cve.cve_id, 'severity': cve.severity.value,
                        'cvss': cve.cvss_score, 'fixed_in': cve.fixed_version,
                        'description': cve.description[:60] + '...'
                    })
    findings.sort(key=lambda x: x['cvss'], reverse=True)
    return findings

all_deps = get_all_dependencies(DEPENDENCY_GRAPH)
print(f'Direct dependencies: {len(DEPENDENCY_GRAPH["myapp"])}')
print(f'Total transitive: {len(all_deps)}')

findings = scan_dependencies(DEPENDENCY_GRAPH)
print(f'\nVulnerable packages: {len(findings)}')
print(f'{"Package":20s} {"Version":10s} {"CVE":20s} {"CVSS":6s} {"Severity"}')
print('-'*75)
for f in findings:
    print(f'{f["package"]:20s} {f["version"]:10s} {f["cve"]:20s} {f["cvss"]:6.1f} {f["severity"]}')


## CI Gate & Remediation Report

In [ ]:
def dependency_ci_gate(findings: List[Dict], fail_on: List[str] = None) -> bool:
    if fail_on is None:
        fail_on = ['CRITICAL', 'HIGH']

    blocking = [f for f in findings if f['severity'] in fail_on]

    print('=== Dependency Scan CI Gate ===')
    print(f'Total findings: {len(findings)}')
    print(f'Blocking ({"|".join(fail_on)}): {len(blocking)}')

    if blocking:
        print('\nRequired fixes:')
        for f in blocking:
            fix = f['fixed_in'] if f['fixed_in'] else 'No fix available'
            print(f'  pip install "{f["package"]}>={fix}"  # fixes {f["cve"]}')

    result = 'PASS' if len(blocking) == 0 else 'FAIL'
    print(f'\nCI Gate: {result}')
    return len(blocking) == 0

dependency_ci_gate(findings)
